# Tourism Package Purchase Prediction – End-to-End MLOps Pipeline

This notebook implements an end-to-end machine learning and MLOps workflow to predict whether a customer will purchase the Wellness Tourism Package using the provided tourism dataset. It follows the project rubric: data registration on Hugging Face, data preparation, model building with experiment tracking, model registration, deployment using a Streamlit app on Hugging Face Spaces, and a CI/CD pipeline using GitHub Actions.


## 1. Environment setup and repository structure

In this section we:
- Install and import the required Python packages.
- Create a clear repository-style folder structure suitable for pushing to GitHub.

> **Note:** Run the setup cell once at the start of your session. If you are using Google Colab, make sure the runtime has internet access for installing packages.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# If running in a fresh environment (e.g. Colab), uncomment the next cell to install dependencies
# !pip install -q pandas numpy scikit-learn joblib datasets huggingface_hub streamlit

import os
import pandas as pd
import numpy as np

from pathlib import Path

# Create base repo-like structure
base_dirs = [
    'data/raw',
    'data/processed',
    'notebooks',
    'src/utils',
    'src/data',
    'src/models',
    'src/deployment',
    'experiments',
    '.github/workflows'
]

for d in base_dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Created/verified folder structure:")
for d in base_dirs:
    print("-", d)


## 2. Local data inspection (tourism.csv)

We first load the provided `tourism.csv` file to understand its schema, basic statistics, and the target variable `ProdTaken` (1 = purchased package, 0 = did not purchase). This step is local exploration; the official pipeline will use the same data after registering it on Hugging Face Datasets.


In [ ]:
# Adjust the path if your tourism.csv is located elsewhere
local_csv_path = 'data/raw/tourism.csv'

# If the file is not yet in data/raw, but exists in the current directory, move/copy it
if not os.path.exists(local_csv_path) and os.path.exists('tourism.csv'):
    os.replace('tourism.csv', local_csv_path)

assert os.path.exists(local_csv_path), 'tourism.csv not found. Place it in data/raw or the working directory.'

raw_df = pd.read_csv(local_csv_path)
print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
raw_df.head()


## 3. Data registration on Hugging Face Datasets (Rubric: Data Registration)

In this section we:
- Authenticate with Hugging Face using a **write** token.
- Create a Hugging Face **dataset repository** (if it does not exist).
- Upload `tourism.csv` to the dataset repo.

Before running the code:
- Create a Hugging Face account.
- Generate a write-access token from **Settings → Access Tokens**.
- Set an environment variable `HF_TOKEN` with this token (for example in a `.env` file or Colab secret).


In [ ]:
from huggingface_hub import HfApi, HfFolder, create_repo, upload_file

HF_TOKEN = os.environ.get('HF_TOKEN')
DATASET_REPO_ID = 'your-username/tourism-wellness-dataset'  # TODO: change to your HF username and desired dataset name

assert HF_TOKEN is not None, 'Please set HF_TOKEN as an environment variable before running.'

HfFolder.save_token(HF_TOKEN)
api = HfApi()

# Create the dataset repo if it does not exist
create_repo(
    repo_id=DATASET_REPO_ID,
    token=HF_TOKEN,
    repo_type='dataset',
    exist_ok=True,
)

# Upload the raw tourism.csv into the dataset repo
upload_file(
    path_or_fileobj=local_csv_path,
    path_in_repo='tourism.csv',
    repo_id=DATASET_REPO_ID,
    repo_type='dataset',
    token=HF_TOKEN,
)

print(f'Registered dataset at: https://huggingface.co/datasets/{DATASET_REPO_ID}')


## 4. Data preparation (Rubric: Data Preparation)

In this section we:
- Load the dataset directly from the Hugging Face dataset space.
- Perform basic data cleaning (e.g., dropping unnecessary index columns, trimming strings, handling missing values).
- Split the cleaned data into train and test sets.
- Save `train.csv` and `test.csv` locally and upload them back to the Hugging Face dataset space.


In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Load the raw data from the Hugging Face dataset repository
hf_ds = load_dataset(DATASET_REPO_ID, data_files={'full': 'tourism.csv'})['full']
df = hf_ds.to_pandas()

print('Shape from HF dataset:', df.shape)
print('Columns from HF dataset:', df.columns.tolist())

def clean_data(df):
    # Drop auto-generated index column if present
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    # Strip whitespace from common categorical columns
    for col in ['Gender', 'Occupation', 'TypeofContact', 'ProductPitched', 'MaritalStatus', 'Designation']:
        if col in df.columns and df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip()

    # Drop rows where target is missing
    df = df.dropna(subset=['ProdTaken'])

    # Simple missing value strategy: fill numeric NaN with median, categorical NaN with mode
    for col in df.columns:
        if df[col].dtype != 'object':
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

    return df

clean_df = clean_data(df)
print('Shape after cleaning:', clean_df.shape)
clean_df.head()


In [ ]:
# Train-test split with stratification on target ProdTaken
TARGET_COL = 'ProdTaken'

X = clean_df.drop(columns=[TARGET_COL])
y = clean_df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

train_df = X_train.copy()
train_df[TARGET_COL] = y_train

test_df = X_test.copy()
test_df[TARGET_COL] = y_test

train_path = 'data/processed/train.csv'
test_path = 'data/processed/test.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)


In [ ]:
# Upload the processed train and test splits back to the HF dataset space
upload_file(
    path_or_fileobj=train_path,
    path_in_repo='train.csv',
    repo_id=DATASET_REPO_ID,
    repo_type='dataset',
    token=HF_TOKEN,
)

upload_file(
    path_or_fileobj=test_path,
    path_in_repo='test.csv',
    repo_id=DATASET_REPO_ID,
    repo_type='dataset',
    token=HF_TOKEN,
)

print('Uploaded train.csv and test.csv to HF dataset repo.')


## 5. Model building with experiment tracking (Rubric: Model Building with Experimentation Tracking)

We now:
- Load `train.csv` and `test.csv` from the Hugging Face data space.
- Build a preprocessing + model pipeline.
- Perform hyperparameter tuning using cross-validation.
- Log the best parameters and evaluation metrics.
- Save the best model locally for later registration on the Hugging Face model hub.

We will use a `GradientBoostingClassifier` (but you could replace it with Decision Tree, Random Forest, XGBoost, etc.).


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import joblib

# Reload splits from HF dataset repo to mimic production setting
train_ds = load_dataset(DATASET_REPO_ID, data_files={'train': 'train.csv'})['train']
test_ds = load_dataset(DATASET_REPO_ID, data_files={'test': 'test.csv'})['test']

train_df = train_ds.to_pandas()
test_df = test_ds.to_pandas()

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]
X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print('Train shape (from HF):', train_df.shape)
print('Test shape (from HF):', test_df.shape)


In [ ]:
# Define categorical and numerical columns based on the data dictionary
categorical_cols = [
    'TypeofContact',
    'Occupation',
    'Gender',
    'ProductPitched',
    'MaritalStatus',
    'Designation',
]

# Treat CityTier as numeric or categorical; here we use numeric
numeric_cols = [
    'Age',
    'CityTier',
    'DurationOfPitch',
    'NumberOfPersonVisiting',
    'NumberOfFollowups',
    'PreferredPropertyStar',
    'NumberOfTrips',
    'MonthlyIncome',
    'Passport',
    'PitchSatisfactionScore',
    'OwnCar',
    'NumberOfChildrenVisiting',
]

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

categorical_transformer = Pipeline(
    steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

clf = GradientBoostingClassifier(random_state=42)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('clf', clf),
    ]
)

param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_depth': [3, 4],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2,
)

grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)


In [ ]:
best_model = grid_search.best_estimator_

# Evaluate on test set
y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

print(f'Test AUC: {auc:.4f}')
print(f'Test F1 : {f1:.4f}')
print(f'Test Acc: {acc:.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Wellness Package Prediction')
plt.show()

# Save the best model locally
os.makedirs('models', exist_ok=True)
model_path = 'models/best_model.pkl'
joblib.dump(best_model, model_path)
print('Saved best model to', model_path)


In [ ]:
# Simple CSV-based experiment tracking: log best params and metrics
import csv

os.makedirs('experiments', exist_ok=True)
log_path = 'experiments/experiment_log.csv'

log_row = {
    **{k: v for k, v in grid_search.best_params_.items()},
    'auc': auc,
    'f1': f1,
    'accuracy': acc,
}

write_header = not os.path.exists(log_path)
with open(log_path, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=log_row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print('Logged experiment to', log_path)


## 6. Model registration on Hugging Face Hub (Rubric: Model Building with Experimentation Tracking)

We now register the best-performing model on the Hugging Face model hub. This allows the deployment layer (Streamlit app on Hugging Face Spaces) to download and use the latest model directly from the hub.


In [ ]:
MODEL_REPO_ID = 'your-username/tourism-wellness-model'  # TODO: update with your HF username

create_repo(
    repo_id=MODEL_REPO_ID,
    token=HF_TOKEN,
    repo_type='model',
    exist_ok=True,
)

upload_file(
    path_or_fileobj=model_path,
    path_in_repo='best_model.pkl',
    repo_id=MODEL_REPO_ID,
    repo_type='model',
    token=HF_TOKEN,
)

print(f'Registered model at: https://huggingface.co/{MODEL_REPO_ID}')


## 7. Deployment design – Streamlit app and Dockerfile (Rubric: Model Deployment)

To deploy the model, we:
- Implement a Streamlit application that loads the model from the Hugging Face model hub and takes user inputs.
- Prepare a `Dockerfile` describing how to run the app (for rubric completeness), even though Hugging Face Spaces can run Streamlit apps without a Dockerfile.

The following cell writes the Streamlit app to `src/deployment/app.py`. You can copy this file into a Hugging Face Space (Streamlit SDK) along with a suitable `requirements.txt`.


In [ ]:
app_code = '''import os
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

MODEL_REPO_ID = "your-username/tourism-wellness-model"  # TODO: update
MODEL_FILENAME = "best_model.pkl"

@st.cache_resource
def load_model():
    local_model_path = hf_hub_download(
        repo_id=MODEL_REPO_ID,
        repo_type="model",
        filename=MODEL_FILENAME,
    )
    model = joblib.load(local_model_path)
    return model

def main():
    st.title("Wellness Tourism Package Purchase Prediction")

    st.markdown(
        "Predict whether a customer is likely to purchase the Wellness Tourism Package based on their profile and interaction data."
    )

    age = st.number_input("Age", min_value=18, max_value=80, value=35)
    typeof_contact = st.selectbox("Type of Contact", ["Self Enquiry", "Company Invited"])
    city_tier = st.selectbox("City Tier", [1, 2, 3])
    duration_of_pitch = st.number_input("Duration of Pitch (minutes)", min_value=0, max_value=60, value=10)
    occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Free Lancer", "Large Business"])
    gender = st.selectbox("Gender", ["Male", "Female"])
    num_person = st.number_input("Number of Persons Visiting", min_value=1, max_value=10, value=2)
    num_followups = st.number_input("Number of Follow-ups", min_value=0, max_value=10, value=3)
    product_pitched = st.selectbox("Product Pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"])
    preferred_star = st.selectbox("Preferred Property Star", [1, 2, 3, 4, 5])
    marital_status = st.selectbox("Marital Status", ["Single", "Married", "Divorced", "Unmarried"])
    num_trips = st.number_input("Number of Trips per Year", min_value=0, max_value=20, value=2)
    passport = st.selectbox("Has Passport?", [0, 1])
    pitch_score = st.selectbox("Pitch Satisfaction Score", [1, 2, 3, 4, 5])
    own_car = st.selectbox("Owns Car?", [0, 1])
    num_children = st.number_input("Number of Children Visiting", min_value=0, max_value=10, value=0)
    designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
    monthly_income = st.number_input("Monthly Income", min_value=0, max_value=200000, value=20000)

    if st.button("Predict"):
        model = load_model()
        input_df = pd.DataFrame([
            {
                "Age": age,
                "TypeofContact": typeof_contact,
                "CityTier": city_tier,
                "DurationOfPitch": duration_of_pitch,
                "Occupation": occupation,
                "Gender": gender,
                "NumberOfPersonVisiting": num_person,
                "NumberOfFollowups": num_followups,
                "ProductPitched": product_pitched,
                "PreferredPropertyStar": preferred_star,
                "MaritalStatus": marital_status,
                "NumberOfTrips": num_trips,
                "Passport": passport,
                "PitchSatisfactionScore": pitch_score,
                "OwnCar": own_car,
                "NumberOfChildrenVisiting": num_children,
                "Designation": designation,
                "MonthlyIncome": monthly_income,
            }
        ])

        prob = model.predict_proba(input_df)[0, 1]
        label = "LIKELY to purchase" if prob >= 0.5 else "Unlikely to purchase"
        st.metric("Prediction", label, f"{prob:.2%}")

if __name__ == "__main__":
    main()
'''

with open('src/deployment/app.py', 'w') as f:
    f.write(app_code)

print('Wrote Streamlit app to src/deployment/app.py')


In [ ]:
# Write a Dockerfile for rubric completeness

dockerfile_contents = '''FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY src/ src/
ENV PYTHONPATH=/app/src

EXPOSE 7860

CMD ["streamlit", "run", "src/deployment/app.py", "--server.port=7860", "--server.address=0.0.0.0"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_contents)

print('Wrote Dockerfile to project root')


## 8. MLOps pipeline with GitHub Actions (Rubric: MLOps Pipeline with GitHub Actions Workflow)

This section defines a GitHub Actions workflow that:
- Sets up Python and installs dependencies.
- Runs data registration and preparation scripts.
- Trains and evaluates the model.
- (Optionally) commits experiment logs and updated models back to the main branch.

The following cell writes `.github/workflows/pipeline.yml` which you can review and push to GitHub.


In [ ]:
pipeline_yaml = '''name: mlops-pipeline

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  mlops:
    runs-on: ubuntu-latest

    env:
      HF_TOKEN: ${{ secrets.HF_TOKEN }}

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Data registration (HF dataset)
        run: |
          python src/data/register_data_hf.py

      - name: Data preparation
        run: |
          python src/data/prepare_data.py

      - name: Train and evaluate model
        run: |
          python src/models/train.py

      - name: Commit experiment logs and models
        if: github.ref == 'refs/heads/main'
        run: |
          git config user.name "github-actions"
          git config user.email "actions@github.com"
          git add experiments/ models/
          git commit -m "Update models and experiment logs [skip ci]" || echo "No changes to commit"
          git push origin main || echo "No changes to push"
'''

workflow_path = '.github/workflows/pipeline.yml'
with open(workflow_path, 'w') as f:
    f.write(pipeline_yaml)

print('Wrote GitHub Actions workflow to', workflow_path)


## 9. Helper scripts for pipeline automation

For the GitHub Actions workflow to run end-to-end outside the notebook, we create three Python scripts:
- `src/data/register_data_hf.py` – data registration on Hugging Face.
- `src/data/prepare_data.py` – data cleaning, splitting, and train/test upload.
- `src/models/train.py` – model training, evaluation, experiment logging, and model registration.

The following cells generate these helper scripts based on the same logic used earlier in the notebook.


In [ ]:
register_script = '''import os
from huggingface_hub import HfApi, HfFolder, create_repo, upload_file

HF_TOKEN = os.environ.get("HF_TOKEN")
DATASET_REPO_ID = "your-username/tourism-wellness-dataset"  # TODO: update

def main():
    assert HF_TOKEN is not None, "HF_TOKEN env var must be set"

    HfFolder.save_token(HF_TOKEN)
    api = HfApi()

    create_repo(
        repo_id=DATASET_REPO_ID,
        token=HF_TOKEN,
        repo_type="dataset",
        exist_ok=True,
    )

    local_csv_path = os.path.join("data", "raw", "tourism.csv")
    assert os.path.exists(local_csv_path), f"{local_csv_path} not found"

    upload_file(
        path_or_fileobj=local_csv_path,
        path_in_repo="tourism.csv",
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )

if __name__ == "__main__":
    main()
'''

with open('src/data/register_data_hf.py', 'w') as f:
    f.write(register_script)

print('Wrote src/data/register_data_hf.py')


In [ ]:
prepare_script = '''import os
import pandas as pd
from datasets import load_dataset
from huggingface_hub import upload_file
from sklearn.model_selection import train_test_split

HF_TOKEN = os.environ.get("HF_TOKEN")
DATASET_REPO_ID = "your-username/tourism-wellness-dataset"  # TODO: update
TARGET_COL = "ProdTaken"


def clean_data(df):
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    for col in ["Gender", "Occupation", "TypeofContact", "ProductPitched", "MaritalStatus", "Designation"]:
        if col in df.columns and df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

    df = df.dropna(subset=[TARGET_COL])

    for col in df.columns:
        if df[col].dtype != "object":
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

    return df


def main():
    ds = load_dataset(DATASET_REPO_ID, data_files={"full": "tourism.csv"})["full"]
    df = ds.to_pandas()

    df = clean_data(df)

    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    train_df = X_train.copy()
    train_df[TARGET_COL] = y_train

    test_df = X_test.copy()
    test_df[TARGET_COL] = y_test

    os.makedirs("data/processed", exist_ok=True)
    train_path = os.path.join("data", "processed", "train.csv")
    test_path = os.path.join("data", "processed", "test.csv")
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    upload_file(
        path_or_fileobj=train_path,
        path_in_repo="train.csv",
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    upload_file(
        path_or_fileobj=test_path,
        path_in_repo="test.csv",
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )

if __name__ == "__main__":
    main()
'''

with open('src/data/prepare_data.py', 'w') as f:
    f.write(prepare_script)

print('Wrote src/data/prepare_data.py')


In [ ]:
train_script = '''import os
import joblib
import pandas as pd
from datasets import load_dataset
from huggingface_hub import create_repo, upload_file
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.model_selection import GridSearchCV

HF_TOKEN = os.environ.get("HF_TOKEN")
DATASET_REPO_ID = "your-username/tourism-wellness-dataset"  # TODO: update
MODEL_REPO_ID = "your-username/tourism-wellness-model"      # TODO: update
TARGET_COL = "ProdTaken"


def load_splits():
    train_ds = load_dataset(DATASET_REPO_ID, data_files={"train": "train.csv"})["train"]
    test_ds = load_dataset(DATASET_REPO_ID, data_files={"test": "test.csv"})["test"]
    train_df = train_ds.to_pandas()
    test_df = test_ds.to_pandas()
    return train_df, test_df


def build_model():
    categorical_cols = [
        "TypeofContact",
        "Occupation",
        "Gender",
        "ProductPitched",
        "MaritalStatus",
        "Designation",
    ]

    numeric_cols = [
        "Age",
        "CityTier",
        "DurationOfPitch",
        "NumberOfPersonVisiting",
        "NumberOfFollowups",
        "PreferredPropertyStar",
        "NumberOfTrips",
        "MonthlyIncome",
        "Passport",
        "PitchSatisfactionScore",
        "OwnCar",
        "NumberOfChildrenVisiting",
    ]

    numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
    categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    clf = GradientBoostingClassifier(random_state=42)

    model = Pipeline(steps=[("preprocessor", preprocessor), ("clf", clf)])
    return model


def train_and_evaluate():
    train_df, test_df = load_splits()

    X_train = train_df.drop(columns=[TARGET_COL])
    y_train = train_df[TARGET_COL]
    X_test = test_df.drop(columns=[TARGET_COL])
    y_test = test_df[TARGET_COL]

    model = build_model()

    param_grid = {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_depth": [3, 4],
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=3,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=2,
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_proba = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    os.makedirs("models", exist_ok=True)
    model_path = os.path.join("models", "best_model.pkl")
    joblib.dump(best_model, model_path)

    os.makedirs("experiments", exist_ok=True)
    log_path = os.path.join("experiments", "experiment_log.csv")
    import csv

    row = {**grid.best_params_, "auc": auc, "f1": f1, "accuracy": acc}
    write_header = not os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    return model_path


def register_model(model_path: str):
    create_repo(
        repo_id=MODEL_REPO_ID,
        token=HF_TOKEN,
        repo_type="model",
        exist_ok=True,
    )

    upload_file(
        path_or_fileobj=model_path,
        path_in_repo="best_model.pkl",
        repo_id=MODEL_REPO_ID,
        repo_type="model",
        token=HF_TOKEN,
    )


def main():
    model_path = train_and_evaluate()
    register_model(model_path)


if __name__ == "__main__":
    main()
'''

with open('src/models/train.py', 'w') as f:
    f.write(train_script)

print('Wrote src/models/train.py')


## 10. Output evaluation (Rubric: Output Evaluation)

Once you have pushed this project to GitHub and deployed the Space, update this section with:

- **GitHub repository link** for this project.
- **Hugging Face Dataset link** for the tourism data.
- **Hugging Face Model link** for the registered model.
- **Hugging Face Space link (Streamlit app)** showing the deployed frontend.

You can also paste screenshots (or markdown image links) for:
- The GitHub Actions workflow run (green checkmark, logs showing steps).
- The GitHub folder structure.
- The running Streamlit app in the Hugging Face Space.


## 11. Conclusion and next steps (Rubric: Notebook Overall Quality)

In this notebook we implemented a complete MLOps workflow for the Wellness Tourism Package prediction problem:
- Registered the dataset and processed train/test splits on Hugging Face Datasets.
- Built and tuned a supervised learning model to predict `ProdTaken`.
- Logged best parameters and evaluation metrics for experiment tracking.
- Registered the best model on the Hugging Face model hub.
- Generated deployment artifacts for a Streamlit app on Hugging Face Spaces.
- Defined a GitHub Actions pipeline to automate the end-to-end workflow.

**Next steps** could include:
- Trying more advanced models (e.g., XGBoost, LightGBM) and comparing performance.
- Adding more robust monitoring and alerting for model performance drift.
- Extending the UI and adding explanations (e.g., feature importance) for business users.
